In [ ]:
#lOAD PACKAGES
import os
import pandas as pd
import numpy as np
import sys
import plotly.express as px
from plotly.subplots import make_subplots
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, combine_solutions, apply_conservative_classification
import sys
sys.path.append('..')
from model.utils.e_reserve_mu import calculate_mu
# from .model.utils.e_reserve_mu import calculate_mu
# 
# from ..model.utils.e_reserve_mu import calculate_mu

idx = pd.IndexSlice



In [ ]:



ss = [
    # {'solution_folder': f"RTS-GMLC_v3.1.1s", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v4.1.1s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v_s1.2s", 'VLGEN': 1e3, 'model_type' : 'stochastic'}
]
days = [131, 320]
s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
solution_keys = ['demand','reserve', 'energy_reserve']
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_ed_name = 's_sed'
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

    # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

    # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

    # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)
# gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
# gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

for k,v in s_uc.items():
    if 'µ' in v.columns:
        s_uc[k] = apply_conservative_classification(s_uc[k])
for k,v in s_ed.items():
    if 'µ' in v.columns:
        s_ed[k]= apply_conservative_classification(s_ed[k])

# if 'µ' in gcdi_KPI_adequacy.columns: 
# #         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
#     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
#     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)

# storage_ids = range(101,111)
# for key in s_uc.keys():
#     s_uc[key] = s_uc[key][s_uc[key].r_id.isin(storage_ids)]
# for key in s_ed.keys():    
#     s_ed[key] = s_ed[key][s_ed[key].r_id.isin(storage_ids)] 



In [ ]:
demand_uc = s_uc['demand'][['demand_MW', 'day','hour','model_type']]
demand_uc.set_index(['day','hour','model_type'], inplace=True)
demand_ed = s_ed['demand'][['demand_MW', 'day','hour','model_type','iteration']]
demand_ed.set_index(['day','hour','model_type','iteration'], inplace=True)
# Calculate the difference
# demand_ed = demand_ed.loc[idx[:,:,demand_ed.index.get_level_values('model_type').unique()[0],demand_ed.index.get_level_values('iteration').unique()[0]]]

imbalance = demand_ed - demand_uc.loc[demand_ed.index.droplevel('iteration')].values # Broadcast demand_uc to match demand_ed's index
imbalance = imbalance.loc[idx[:,:,imbalance.index.get_level_values('model_type').unique()[0],imbalance.index.get_level_values('iteration').unique()[0]]]
##
tuples = [(d, h_i, h) for d, h in imbalance.index for h_i in imbalance.index.get_level_values('hour').unique() if h_i <= h]
cum_imbalance = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   

cum_imbalance = cum_imbalance.merge(imbalance, left_on=['day','hour'], right_on=['day', 'hour']) 
# # cum_imbalance.set_index(['day','hour_i', 'model_type', 'iteration'], inplace=True)
cum_imbalance.set_index(['day', 'hour_i', 'hour'], inplace=True)
cum_imbalance = cum_imbalance.groupby(['day', 'hour_i']).cumsum()
cum_imbalance['positive_MW'] = cum_imbalance.demand_MW.clip(lower=0)
cum_imbalance['negative_MW'] = cum_imbalance.demand_MW.clip(upper=0).abs()

# cum_imbalance = cum_imbalance.loc[idx[:,:,:,cum_imbalance.index.get_level_values('model_type').unique()[0],cum_imbalance.index.get_level_values('iteration').unique()[0]]]

In [ ]:
reserve = s_uc['reserve'][['day','hour','model_type','required_reserve_up_MW', 'required_reserve_down_MW']].set_index(['day','hour','model_type'])
reserve = reserve.loc[idx[:,:,reserve.index.get_level_values('model_type').unique()[0]]]
reserve = reserve.dropna()  # Drop any rows with NA values

tuples = [(d, h_i, h) for d, h in reserve.index for h_i in reserve.index.get_level_values('hour').unique() if h_i <= h]
cum_reserve = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
cum_reserve = cum_reserve.merge(reserve, left_on=['day','hour'], right_on=['day','hour']) 
cum_reserve.set_index(['day','hour_i', 'hour'], inplace=True)
cum_reserve = cum_reserve.groupby(['day','hour_i']).cumsum()

In [ ]:
energy_reserve = s_uc['energy_reserve'][['day','hour_i', 'hour','model_type','required_energy_reserve_up_MW', 'required_energy_reserve_down_MW']].set_index(['day', 'hour_i', 'hour', 'model_type'])
energy_reserve = energy_reserve.loc[idx[:,:,:,energy_reserve.index.get_level_values('model_type').unique()[0]]]
energy_reserve = energy_reserve.dropna()  # Drop any rows with NA values

In [ ]:
px.line(
    pd.merge(energy_reserve.groupby(['day','hour']).max(),energy_reserve.loc[:,1,:], suffixes = ('_max','_1'),on = ['day','hour']).reset_index(), 
    x = 'hour',
    y = ['required_energy_reserve_up_MW_max', 'required_energy_reserve_down_MW_max', 'required_energy_reserve_up_MW_1', 'required_energy_reserve_down_MW_1'],
    markers = True,
    facet_col='day',
    # facet_row = 'model_type',
    # title='Max energy reserve requirement over the day',
)


In [ ]:
px.line(cum_reserve.loc[:,1,:].reset_index(),
    x = 'hour',
    y = ['required_reserve_up_MW', 'required_reserve_down_MW'],
    markers = True,
    facet_col='day',
    # facet_row='model_type',
    # title='Max energy reserve requirement over the day',
)

In [ ]:
reserve

In [ ]:
reserve.rename(columns={
        'energy_reserve_up_MW': 'reserve_up_MW',
        'energy_reserve_down_MW': 'reserve_down_MW'
        }).reset_index()

In [ ]:
mu  = calculate_mu(
    reserve.rename(columns={
        'required_reserve_up_MW': 'reserve_up_MW',
        'required_reserve_down_MW': 'reserve_down_MW'
        }).reset_index(),
    energy_reserve.rename(columns={
        'required_energy_reserve_up_MW': 'energy_reserve_up_MW',
        'required_energy_reserve_down_MW': 'energy_reserve_down_MW'
    }).reset_index())

mu.rename(columns={'mu_up': 'required_reserve_up_MW', 'mu_down': 'required_reserve_down_MW'}, inplace=True)

In [ ]:
# days = reserve.index.get_level_values('day').unique()
# mu_list = []



# for day in days:
#     day_reserve = reserve.loc[day]
#     day_energy_reserve = energy_reserve.loc[day]
    
#     n = 24
    
#     # Create matrix A as a lower triangular matrix for up reserves
#     A = np.zeros((n,n))
#     for i in range(n):
#         for j in range(n):
#             if i >= j:  # Lower triangular condition
#                 A[i,j] = day_reserve.iloc[j].required_reserve_up_MW

#     # Get vector B as the maximum energy reserve requirement for each hour
#     # B = day_energy_reserve.groupby('hour').max().required_energy_reserve_up_MW.values
#     B = day_energy_reserve.loc[1,:].required_energy_reserve_up_MW.values
#     # Solve the system
#     x = np.linalg.solve(A, B)
#     mu_up = pd.DataFrame({'mu_up': x}, index=day_reserve.index)
    
#     # Repeat for down reserves
#     A_down = np.zeros((n,n))
#     for i in range(n):
#         for j in range(n):
#             if i >= j:
#                 A_down[i,j] = day_reserve.iloc[j].required_reserve_down_MW
                
#     # B_down = day_energy_reserve.groupby('hour').max().required_energy_reserve_down_MW.values
#     B_down = day_energy_reserve.loc[1,:].required_energy_reserve_down_MW.values
#     x_down = np.linalg.solve(A_down, B_down)
    
#     mu_down = pd.DataFrame({'mu_down': x_down}, index=day_reserve.index)
    
#     # Combine into final DataFrame for this day
#     day_mu = pd.DataFrame({
#         'required_reserve_up_MW': mu_up.mu_up,
#         'required_reserve_down_MW': mu_down.mu_down
#     }, index=day_reserve.index)
#     day_mu['day'] = day
#     mu_list.append(day_mu)

# # Combine all days
# mu = pd.concat(mu_list)
# mu = mu.reset_index().set_index(['day','hour'])
# # mu.rename(columns={'required_reserve_up_MW': 'mu_up', 'required_reserve_down_MW': 'mu_down'}, inplace=True)


In [ ]:

fig = px.line(mu.reset_index(),
    x = 'hour',
    y = ['required_reserve_up_MW', 'required_reserve_down_MW'],
    markers = True,
    facet_col='day',
    # facet_row='model_type',
    # title='Max energy reserve requirement over the day',
)
legend_attr = dict(
    x=0.5,
    y=-0.35,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig.update_layout(legend=legend_attr, height=500, width=800)
fig.show()

In [ ]:
weighted_reserve = (reserve*mu)
tuples = [(d, h_i, h) for d, h in weighted_reserve.index for h_i in weighted_reserve.index.get_level_values('hour').unique() if h_i <= h]
cum_weighted_reserve = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
cum_weighted_reserve = cum_weighted_reserve.merge(weighted_reserve, left_on=['day','hour'], right_on=['day','hour']) 
cum_weighted_reserve.set_index(['day','hour_i', 'hour'], inplace=True)
cum_weighted_reserve = cum_weighted_reserve.groupby(['day','hour_i']).cumsum()

In [ ]:
cum_weighted_reserve.loc[:,1,:]

In [ ]:
cum_weighted_reserve.loc[:,1,:]

In [ ]:
to_plot = pd.concat(
    [cum_weighted_reserve.loc[:,1,:].rename(columns={'required_reserve_up_MW': 'cum_weighted_required_reserve_up_MW', 'required_reserve_down_MW': 'cum_weighted_required_reserve_down_MW'}),
    energy_reserve.loc[:,1,:],
     cum_reserve.loc[:,1,:].rename(columns={'required_reserve_up_MW': 'cum_required_reserve_up_MW', 'required_reserve_down_MW': 'cum_required_reserve_down_MW'})],
    axis = 1).reset_index()
fig = px.line(to_plot,
    x = 'hour',
    y = ['cum_required_reserve_up_MW', 'cum_required_reserve_down_MW', 'cum_weighted_required_reserve_up_MW', 'cum_weighted_required_reserve_down_MW', 'required_energy_reserve_up_MW', 'required_energy_reserve_down_MW'],
    markers = True,
    facet_col='day',
    # facet_row='model_type',
    # title='Max energy reserve requirement over the day',
)
legend_attr = dict(
    x=0.5,
    y=-0.35,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig.update_layout(legend=legend_attr, height=500, width=800)
fig.show()

In [ ]:
# Calculate all ratios and combine them into a single DataFrame
ratios = pd.DataFrame({
    'cum_reserve_vs_energy_up': cum_reserve.required_reserve_up_MW.div(energy_reserve.required_energy_reserve_up_MW),
    'cum_reserve_vs_energy_down': cum_reserve.required_reserve_down_MW.div(energy_reserve.required_energy_reserve_down_MW),
    'cum_weighted_reserve_vs_energy_up': cum_weighted_reserve.required_reserve_up_MW.div(energy_reserve.required_energy_reserve_up_MW),
    'cum_weighted_reserve_vs_energy_down': cum_weighted_reserve.required_reserve_down_MW.div(energy_reserve.required_energy_reserve_down_MW),
})

# Reset the index to make all index levels become columns
ratios = ratios.reset_index()

In [ ]:

to_plot = ratios.melt(id_vars=['day', 'hour_i', 'hour'], value_vars=['cum_reserve_vs_energy_up', 'cum_reserve_vs_energy_down', 'cum_weighted_reserve_vs_energy_up', 'cum_weighted_reserve_vs_energy_down'], var_name='ratio_type', value_name='ratio_value')

fig = px.scatter(
    to_plot,
    y='hour_i',
    x='hour',
    color='ratio_value',
    facet_row='day',
    facet_col='ratio_type',
    color_continuous_scale='Viridis',
    # labels={'ratio_value': 'Probability'},
    # title='P_{i,t}^up (upper graphs) and P_{i,t}^dn (lower graphs)',
)
# fig.show()
# Update the figure with a color scale that emphasizes values > 1


fig.update_layout(
    width=1000,
    height=1000,
    coloraxis=dict(
        cmin=0,
        cmax=1.5,  # Set maximum to 1.5 to highlight values > 1
        colorscale=[
            [0, '#440154'],      # Dark purple for values near 0
            [0.66, '#1f9e89'],   # Teal for values near 1
            [0.8, '#fde725'],    # Yellow for values > 1
            [1, '#ff0000']       # Red for values > 1.2
        ]
    )
)
fig.show()

In [ ]:
legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)
# to_plot = ratios.melt(id_vars=['day', 'hour_i', 'hour'], value_vars=['cum_reserve_vs_energy_up', 'cum_reserve_vs_energy_down'], var_name='ratio_type', value_name='ratio_value')
to_plot = to_plot[(to_plot.ratio_type == 'cum_reserve_vs_energy_up') | (to_plot.ratio_type == 'cum_reserve_vs_energy_down')]
to_plot['ratio_type'] = to_plot['ratio_type'].replace({
    'cum_reserve_vs_energy_up': 'ratio up',
    'cum_reserve_vs_energy_down': 'ratio down'
})
to_plot= to_plot[to_plot.day == to_plot.day.unique()[0]]
scale = 0.7
dim = (1000*scale,500*scale)
fig = px.scatter(
    to_plot,
    y='hour_i',
    x='hour',
    color='ratio_value',
    # facet_row='day',
    facet_col='ratio_type',
    color_continuous_scale='Viridis',
    labels={'ratio_value': 'ratio value', 'hour_i': 'i', 'hour': 't'},
    # title='P_{i,t}^up (upper graphs) and P_{i,t}^dn (lower graphs)',
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
    width=dim[0],
    height=dim[1],
    legend=legend_attr  

)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)
# fig.update_layout(
#     coloraxis=dict(
#         cmin=0,
#         cmax=1.5,  # Set maximum to 1.5 to highlight values > 1
#         colorscale=[
#             [0, '#440154'],      # Dark purple for values near 0
#             [0.66, '#1f9e89'],   # Teal for values near 1
#             [0.8, '#fde725'],    # Yellow for values > 1
#             [1, '#ff0000']       # Red for values > 1.2
#         ]
#     )
# )
# fig.update_layout(coloraxis_showscale=False)
fig.show()


In [ ]:
n = 10
fig.update_layout(margin=dict(l=n, r=n, t=n+20, b=n))
fig.write_image("ratio_cum_reserve_energy_reserve.pdf", width=dim[0], height=dim[1])

In [ ]:
# Calculate all ratios and combine them into a single DataFrame
ratios = pd.DataFrame({
    'imbalance_vs_reserve_down': cum_imbalance.negative_MW.div(cum_reserve.required_reserve_down_MW),
    'imbalance_vs_reserve_up': cum_imbalance.positive_MW.div(cum_reserve.required_reserve_up_MW),
    'imbalance_vs_weighted_reserve_down': cum_imbalance.negative_MW.div(cum_weighted_reserve.required_reserve_down_MW),
    'imbalance_vs_weighted_reserve_up': cum_imbalance.positive_MW.div(cum_weighted_reserve.required_reserve_up_MW),
    'imbalance_vs_energy_down': cum_imbalance.negative_MW.div(energy_reserve.required_energy_reserve_down_MW),
    'imbalance_vs_energy_up': cum_imbalance.positive_MW.div(energy_reserve.required_energy_reserve_up_MW),

})

# Reset the index to make all index levels become columns
ratios = ratios.reset_index()

In [ ]:

to_plot = ratios.melt(id_vars=['day', 'hour_i', 'hour'],
                      value_vars=['imbalance_vs_reserve_down', 'imbalance_vs_reserve_up', 'imbalance_vs_weighted_reserve_down', 'imbalance_vs_weighted_reserve_up', 'imbalance_vs_energy_down', 'imbalance_vs_energy_up'], var_name='ratio_type', value_name='ratio_value')

fig = px.scatter(
    to_plot,
    y='hour_i',
    x='hour',
    color='ratio_value',
    facet_row='day',
    facet_col='ratio_type',
    color_continuous_scale='Viridis',
    # labels={'ratio_value': 'Probability'},
    # title='P_{i,t}^up (upper graphs) and P_{i,t}^dn (lower graphs)',
)
# fig.show()
# Update the figure with a color scale that emphasizes values > 1
fig.update_layout(
    width=1000,
    height=1000,
    coloraxis=dict(
        cmin=0,
        cmax=1.5,  # Set maximum to 1.5 to highlight values > 1
        colorscale=[
            [0, '#440154'],      # Dark purple for values near 0
            [0.66, '#1f9e89'],   # Teal for values near 1
            [0.8, '#fde725'],    # Yellow for values > 1
            [1, '#ff0000']       # Red for values > 1.2
        ]
    )
)
fig.show()
